# Single Agent Team in Autogen

Perfect for tasks where one agent can handle everything but we want a team like-control.

In [1]:
import asyncio
from autogen_agentchat.agents import AssistantAgent

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os


In [2]:
load_dotenv(override=True)

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ API Key loaded successfully.")
else:
    print("❌ API Key not found.")

✅ API Key loaded successfully.


In [3]:
model_client =  OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    model="qwen/qwen-2.5-72b-instruct:free",
    api_key = api_key,
    model_info={
        "family":'deepseek',
        "vision" :True,
        "function_calling":True,
        "json_output": False
    }
)

/Users/lokesh/Autogen/venv/lib/python3.13/site-packages/autogen_ext/models/openai/_openai_client.py:439: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


# Why use a single Agent team?

1. Consistency - termination conditions, logging, observing
2. Scalability - we can add more agents as our use case expands without rewriting logic. 
3. Control - Features like observers or cancellation work seamlessly.

In [4]:
poet = AssistantAgent(
    name = 'solo_poet',
    model_client=model_client,
    system_message="You are a poet. Write a short poem on Moon and end with 'MOONLIGHT FADES'. "
)

termination = TextMentionTermination('MOONLIGHT FADES')

team = RoundRobinGroupChat(
    participants=[poet],
    termination_condition=termination
)

In [5]:
async def test_single_agent():
    task = TextMessage(content="Write a short poem on Moon",source='user')
    result = await team.run(task = task)
    for message in result.messages:
        print(f"{message.source}: {message.content}")


await test_single_agent() 

user: Write a short poem on Moon
solo_poet: Silver orb that rules the night,  
Silent sentinel, bathed in white.  
Casting shadows, soft and deep,  
Guardian of secrets that we keep.  
Whispering tales to those who dream,  
In your glow, all fears seem tame.  
But as the hours gently drift,  
Moonlight fades.
solo_poet: 
solo_poet: MOONLIGHT FADES.


## Single-Agent Team vs. Plain Single Agent

| Aspect             | Single-Agent Team            | Plain Single Agent          |
|--------------------|------------------------------|-----------------------------|
| **Termination**    | Stops with condition (e.g., "MOONLIGHT FADES") | Runs until task ends        |
| **Observability**  | Supports Console observers   | Manual logging only         |
| **Flexibility**    | Easily scales to multi-agent | Fixed to one agent          |